# AI-sfa P3 이펙트 분리 — Kaggle GPU

나카무라 영상의 폭발·연기·파편 구간(157구간, 약 5,500프레임)에서 **이펙트 영역만** 오려 Drive에 올린다.
(Grounding DINO로 '폭발·연기·파편…' 상자를 찾고 SAM 2가 오려 구간 전체로 따라간다. 사람은 뺀다.)

**설정(P1·P2 노트북과 같다):** Accelerator **GPU T4 x2**, Internet **On**, Secrets에 `RCLONE_GDRIVE_AUTH_B64` 붙이기.

**실행:** 오른쪽 위 **Save Version → Save & Run All (Commit)**. 결과: Drive `02_분석자료/이펙트_분리/`.
멈추면 다시 Save & Run All — 끝난 구간은 건너뛴다.

In [ ]:
# 1. 준비: rclone 설치·설정(인증값은 출력하지 않는다), 라이브러리
import os, json, base64, subprocess, pathlib
from kaggle_secrets import UserSecretsClient
raw = UserSecretsClient().get_secret('RCLONE_GDRIVE_AUTH_B64').strip()
d = json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)))
p = pathlib.Path.home() / '.config/rclone'; p.mkdir(parents=True, exist_ok=True)
lines = ['[gdrive]', 'type = drive', 'scope = drive.readonly,drive.file']
if d.get('client_id'): lines.append('client_id = ' + d['client_id'])
if d.get('client_secret'): lines.append('client_secret = ' + d['client_secret'])
lines.append('token = ' + d['token'])
(p / 'rclone.conf').write_text('\n'.join(lines) + '\n'); os.chmod(p / 'rclone.conf', 0o600)
del raw, d, lines
!cd /tmp && curl -sSLo rc.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rc.zip && cp rclone-*/rclone /usr/local/bin/
!rclone lsd gdrive:AI-sfa 2>/dev/null | head -3
!SAM2_BUILD_CUDA=0 pip -q install --no-build-isolation sam2
import torch; print('GPU 수:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2. 작업자 파일 쓰기 + 모델 미리 받기(두 작업자가 동시에 받다 충돌하지 않게)
open('/kaggle/working/kaggle_fx_worker.py', 'w', encoding='utf-8').write("\"\"\"P3 이펙트 분리(Kaggle GPU): 나카무라 이펙트 구간에서 이펙트 영역만 오려 낸다(배경·인물 제외).\n\n사용(노트북이 부른다): python kaggle_fx_worker.py <작업번호> <작업수> [--model large|tiny] [--device cuda|cpu] [--limit N]\n1. 구간 목록: Drive `02_분석자료/이펙트_분리/_segments.csv`(post_id,start,end,tags — 자동 태그 explosion·smoke·debris 등 ≥ 0.35 구간 ±6프레임).\n2. 구간 프레임을 높이 540으로 뽑아 jpg 폴더로 둔다.\n3. Grounding DINO(글자 조건 검출)로 6프레임마다 낱말 묶음 PROMPTS(폭발·연기 계열 / 돌·잔해 계열, 각각 person 포함)로 상자를 찾아 합친다.\n4. SAM 2 영상 예측기에 상자를 그 프레임의 물체로 넣고 구간 전체로 퍼뜨린다(앞뒤 모두).\n5. 프레임별 이펙트 마스크 = 이펙트 물체 합집합 − 사람 물체 합집합.\n   상자 거르기: 화면 80% 이상 상자 버림, 사람은 점수 ≥ PERSON_T이고 이펙트 상자와 IoU ≤ 0.5인 것만(같은 덩어리를 둘 다로 잡는 일이 잦다).\n출력: Drive `02_분석자료/이펙트_분리/<post_id>_<start>_v3.npz` — mask[T,H,W](packbits), frames(원 영상 번호), labels(물체별 글자),\n      obj_mask[물체, T, H, W/8](물체별 packbits — 서버에서 다시 고르기용), area, det(검출 상자 목록).\n   v3: 화면 50% 넘는 상자 버림, 한 프레임에서 화면 30% 넘게 덮는 물체는 기본 합집합(mask)에서 뺌. 원본 영상 유래라 Drive에만 둔다. 끝난 구간은 건너뛴다.\n\"\"\"\nimport argparse\nimport csv\nimport io\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nimport cv2\nimport numpy as np\n\nSRC = \"gdrive:AI-sfa/01_원본영상/yutaka_nakamura_디지털\"\nOUT = \"gdrive:AI-sfa/02_분석자료/이펙트_분리\"\n# 검출 낱말 묶음(2026-10-22 v2): 'debris' 같은 추상 낱말은 흩어진 돌 조각을 못 찾았다(271642: 사람만 검출).\n# 'rocks. stones.'처럼 보이는 물건 이름이면 조각마다 잡혔다(Grounding DINO base, 점수 0.23~0.46). 묶음마다 따로 검출해 합친다.\nPROMPTS = [\"explosion. smoke. dust cloud. fire. shockwave. sparks. flash. person.\",\n           \"rocks. stones. rubble. debris. person.\"]\nVERSION = \"_v3\"\nH = 540\nEVERY = 6\nBOX_T, TEXT_T = 0.22, 0.2\n\n\ndef rc(*a):\n    return subprocess.run([\"rclone\", *a], capture_output=True, text=True)\n\n\ndef log(*a):\n    print(time.strftime(\"%H:%M:%S\"), *a, flush=True)\n\n\ndef frames_to_dir(video, a, b, d):\n    os.makedirs(d, exist_ok=True)\n    cap = cv2.VideoCapture(video)\n    cap.set(cv2.CAP_PROP_POS_FRAMES, a)\n    n = 0\n    for f in range(a, b + 1):\n        ok, im = cap.read()\n        if not ok:\n            break\n        im = cv2.resize(im, (round(im.shape[1] * H / im.shape[0]), H), interpolation=cv2.INTER_AREA)\n        cv2.imwrite(os.path.join(d, f\"{n:05d}.jpg\"), im, [cv2.IMWRITE_JPEG_QUALITY, 95])\n        n += 1\n    return n\n\n\nclass Detector:\n    def __init__(self, device, tiny=False):\n        import torch\n        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor\n        mid = \"IDEA-Research/grounding-dino-tiny\" if tiny else \"IDEA-Research/grounding-dino-base\"\n        self.proc = AutoProcessor.from_pretrained(mid)\n        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(mid).to(device).eval()\n        self.device = device\n        self.torch = torch\n\n    def __call__(self, bgr):\n        out = []\n        for prompt in PROMPTS:\n            out += self.detect(bgr, prompt)\n        return nms(out)\n\n    def detect(self, bgr, prompt):\n        from PIL import Image\n        img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))\n        inp = self.proc(images=img, text=prompt, return_tensors=\"pt\").to(self.device)\n        with self.torch.no_grad():\n            out = self.model(**inp)\n        try:  # transformers 판마다 인수 이름이 다르다\n            r = self.proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=BOX_T, text_threshold=TEXT_T,\n                                                                 target_sizes=[img.size[::-1]])[0]\n        except TypeError:\n            r = self.proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=BOX_T, text_threshold=TEXT_T,\n                                                                 target_sizes=[img.size[::-1]])[0]\n        labels = r.get(\"text_labels\", r.get(\"labels\"))\n        return [(str(l), float(s), [float(v) for v in b]) for l, s, b in zip(labels, r[\"scores\"].tolist(), r[\"boxes\"].tolist())]\n\n\nPERSON_T = 0.4\nBIG_BOX = 0.5\nBIG_OBJ = 0.3  # 한 프레임에서 화면 30% 넘게 덮는 물체는 기본 합집합에서 뺀다(배경 바닥·벽, v2 271642·76453)\n\n\ndef box_area(b):\n    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])\n\n\ndef iou(a, b):\n    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])\n    inter = max(0.0, x1 - x0) * max(0.0, y1 - y0)\n    return inter / max(box_area(a) + box_area(b) - inter, 1e-6)\n\n\ndef nms(dets, thr=0.6):\n    \"\"\"같은 종류(사람/이펙트)끼리 많이 겹치는 상자는 점수 높은 것 하나만.\"\"\"\n    keep = []\n    for d in sorted(dets, key=lambda x: -x[1]):\n        if all(is_fx(d[0]) != is_fx(k[0]) or iou(d[2], k[2]) <= thr for k in keep):\n            keep.append(d)\n    return keep\n\n\ndef is_fx(label):\n    \"\"\"사람이 아닌 검출은 모두 이펙트로 본다('dust cloud'가 'cloud'로 잘려 나오기도 한다).\"\"\"\n    return \"person\" not in label\n\n\ndef run_segment(det, pred, video, a, b, tmp):\n    d = os.path.join(tmp, \"fr\")\n    shutil.rmtree(d, ignore_errors=True)\n    n = frames_to_dir(video, a, b, d)\n    if n == 0:\n        return None\n    dets = []\n    W = cv2.imread(os.path.join(d, \"00000.jpg\")).shape[1]\n    for i in range(0, n, EVERY):\n        im = cv2.imread(os.path.join(d, f\"{i:05d}.jpg\"))\n        found = [(l, s, b) for l, s, b in det(im) if box_area(b) < BIG_BOX * W * H]  # 화면 절반 넘는 상자는 배경을 통째로 잡았다(v2)\n        fxb = [b for l, s, b in found if is_fx(l)]\n        for lab, sc, box in found:\n            # 같은 덩어리를 'explosion'과 'person'으로 둘 다 잡는 일이 잦다(구름이 통째로 지워졌다) → 이펙트 우선\n            if not is_fx(lab) and (sc < PERSON_T or any(iou(box, b) > 0.5 for b in fxb)):\n                continue\n            dets.append((i, lab, sc, box))\n    mask = np.zeros((n, H, W), bool)\n    labels = []\n    objs = {}  # 물체별 마스크(프레임 → 배열). 나중에 서버에서 물체를 다시 고를 수 있게 따로 둔다\n    if any(is_fx(l) for _, l, _, _ in dets):\n        import torch\n        # SAM 2 영상 예측기는 기억 특징을 bfloat16으로 저장한다(autocast 전제). 감싸지 않으면 dtype 오류\n        with torch.inference_mode(), torch.autocast(pred.device.type, dtype=torch.bfloat16):\n            st = pred.init_state(video_path=d, offload_video_to_cpu=True, offload_state_to_cpu=True)\n            for k, (i, lab, sc, box) in enumerate(dets):\n                pred.add_new_points_or_box(st, frame_idx=i, obj_id=k, box=np.array(box, np.float32))\n                labels.append(lab)\n            fxid = np.array([is_fx(l) for l in labels])\n            seen = set()\n            for rev in (False, True):  # 상자를 넣은 프레임 앞쪽으로도 퍼뜨린다\n                for fi, ids, logits in pred.propagate_in_video(st, reverse=rev):\n                    m = (logits[:, 0] > 0).cpu().numpy()\n                    ids = np.asarray(ids)\n                    for j, oid in enumerate(ids):\n                        if m[j].any():\n                            prev = objs.setdefault(int(oid), {}).get(fi)\n                            objs[int(oid)][fi] = m[j] if prev is None else (prev | m[j])\n                    sel = fxid[ids] & (m.reshape(len(ids), -1).mean(1) <= BIG_OBJ)\n                    e = m[sel].any(0) if sel.any() else np.zeros((H, W), bool)\n                    p = m[~sel].any(0) if (~sel).any() else np.zeros((H, W), bool)\n                    v = e & ~p\n                    mask[fi] = (mask[fi] | v) if fi in seen else v\n                    seen.add(fi)\n            pred.reset_state(st)\n    ob = np.zeros((len(labels), n, H, (W + 7) // 8), np.uint8)\n    for oid, fr in objs.items():\n        for fi, mm in fr.items():\n            ob[oid, fi] = np.packbits(mm, axis=-1)\n    return {\"mask\": np.packbits(mask, axis=-1), \"shape\": np.array(mask.shape), \"frames\": np.arange(a, a + n),\n            \"obj_mask\": ob,\n            \"labels\": np.array(labels), \"area\": mask.reshape(n, -1).mean(1).astype(np.float32),\n            \"det\": np.array(json.dumps(dets))}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"wid\", type=int)\n    ap.add_argument(\"nw\", type=int)\n    ap.add_argument(\"--model\", default=\"large\")\n    ap.add_argument(\"--device\", default=\"cuda\")\n    ap.add_argument(\"--limit\", type=int, default=0)\n    ap.add_argument(\"--segments\", default=\"\")\n    a = ap.parse_args()\n    tmp = f\"/tmp/fx{a.wid}\"\n    os.makedirs(tmp, exist_ok=True)\n    seg_csv = a.segments or os.path.join(tmp, \"_segments.csv\")\n    if not a.segments:\n        rc(\"copyto\", f\"{OUT}/_segments.csv\", seg_csv)\n    segs = list(csv.DictReader(open(seg_csv, encoding=\"utf-8\")))\n    done = set(rc(\"lsf\", OUT).stdout.split())\n    todo = [s for s in segs if f\"{s['post_id']}_{s['start']}{VERSION}.npz\" not in done][a.wid::a.nw]\n    if a.limit:\n        todo = todo[:a.limit]\n    log(\"남은 구간\", len(todo))\n    from sam2.sam2_video_predictor import SAM2VideoPredictor\n    pred = SAM2VideoPredictor.from_pretrained(\"facebook/sam2.1-hiera-\" + (\"tiny\" if a.model == \"tiny\" else \"large\"),\n                                              device=a.device)\n    det = Detector(a.device, tiny=a.model == \"tiny\")\n    vids = {os.path.basename(p).split(\"_\")[0]: p for p in rc(\"lsf\", \"--files-only\", \"--include\", \"*.mp4\", SRC).stdout.split()}\n    cur = None\n    t0 = time.time()\n    for k, s in enumerate(todo):\n        try:\n            pid, st, en = s[\"post_id\"], int(s[\"start\"]), int(s[\"end\"])\n            if pid not in vids:\n                log(\"영상 없음\", pid)\n                continue\n            loc = os.path.join(tmp, \"cur.mp4\")\n            if cur != pid:\n                rc(\"copyto\", f\"{SRC}/{vids[pid]}\", loc)\n                cur = pid\n            r = run_segment(det, pred, loc, st, en, tmp)\n            if r is None:\n                continue\n            buf = io.BytesIO()\n            np.savez_compressed(buf, **r)\n            p = os.path.join(tmp, \"out.npz\")\n            open(p, \"wb\").write(buf.getvalue())\n            up = rc(\"copyto\", p, f\"{OUT}/{pid}_{st}{VERSION}.npz\")\n            if up.returncode:\n                log(\"올리기 실패\", up.stderr[-200:])\n            log(k + 1, \"/\", len(todo), pid, st, en, \"물체\", len(r[\"labels\"]), \"이펙트 면적 평균\", round(float(r[\"area\"].mean()), 3),\n                \"누적\", round((time.time() - t0) / 60, 1), \"분\")\n        except Exception:\n            traceback.print_exc()\n    log(\"끝\")\n\n\nif __name__ == \"__main__\":\n    main()\n")
import subprocess
subprocess.run(['python', '-c', "from sam2.sam2_video_predictor import SAM2VideoPredictor as P; P.from_pretrained('facebook/sam2.1-hiera-large', device='cpu'); from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection as M; AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-base'); M.from_pretrained('IDEA-Research/grounding-dino-base')"], check=True)
print('준비 끝')

In [ ]:
# 3. 실행: GPU마다 작업자 하나. 5분마다 진행 상황을 찍는다
import subprocess, time, os
ng = max(1, torch.cuda.device_count())
procs = {}
for g in range(ng):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    procs[g] = subprocess.Popen(['python', '/kaggle/working/kaggle_fx_worker.py', str(g), str(ng)], env=env,
                                stdout=open(f'/kaggle/working/fx{g}.log', 'w'), stderr=subprocess.STDOUT)
t0 = time.time()
while procs and time.time() - t0 < 11.6 * 3600:
    time.sleep(300)
    for g, pr in list(procs.items()):
        lines = [l for l in open(f'/kaggle/working/fx{g}.log', encoding='utf-8', errors='replace').read().splitlines() if '/' in l or '끝' in l or 'Error' in l]
        print(round((time.time() - t0) / 60), '분 GPU', g, '|', ' / '.join(lines[-2:])[-300:], flush=True)
        if pr.poll() is not None:
            del procs[g]
for pr in procs.values(): pr.terminate()
print('종료')